In [6]:
import cv2
import numpy as np
import mediapipe as mp
import matplotlib.pyplot as plt
import os
import urllib.request

# Imports strictly from the new Tasks API
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision import drawing_utils
from mediapipe.tasks.python.vision import drawing_styles

from tqdm import tqdm  # For progress bars in loops

# Allow matplotlib to display inline
%matplotlib inline

class PoseDetector:
    def __init__(self, model_asset_path='pose_landmarker_heavy.task'):
        # Pure Python fallback to download the model if missing
        if not os.path.exists(model_asset_path):
            print(f"Model not found. Downloading to {model_asset_path}...")
            url = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/1/pose_landmarker_heavy.task"
            urllib.request.urlretrieve(url, model_asset_path)
            print("✅ Download complete!")

        # Create the PoseLandmarker object for VIDEO
        base_options = python.BaseOptions(model_asset_path=model_asset_path)
        options = vision.PoseLandmarkerOptions(
            base_options=base_options,
            running_mode=vision.RunningMode.VIDEO,
            min_pose_detection_confidence=0.5,
            min_pose_presence_confidence=0.5,
            min_tracking_confidence=0.5
        )
        self.detector = vision.PoseLandmarker.create_from_options(options)
        
        # Dictionary mapping MediaPipe indices to human-readable names (Excluding Face 0-10)
        self.keypoint_names = {
            11: 'L_Shoulder', 12: 'R_Shoulder', 13: 'L_Elbow', 14: 'R_Elbow',
            15: 'L_Wrist', 16: 'R_Wrist', 17: 'L_Pinky', 18: 'R_Pinky',
            19: 'L_Index', 20: 'R_Index', 21: 'L_Thumb', 22: 'R_Thumb',
            23: 'L_Hip', 24: 'R_Hip', 25: 'L_Knee', 26: 'R_Knee',
            27: 'L_Ankle', 28: 'R_Ankle', 29: 'L_Heel', 30: 'R_Heel',
            31: 'L_Foot', 32: 'R_Foot'
        }

    def find_pose(self, img, timestamp_ms, draw=True):
        """Processes the image and draws landmarks using the new API utilities."""
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)

        self.results = self.detector.detect_for_video(mp_image, int(timestamp_ms))

        if self.results.pose_landmarks and draw:
            annotated_rgb = self._draw_landmarks_on_image(img_rgb, self.results)
            return cv2.cvtColor(annotated_rgb, cv2.COLOR_RGB2BGR)
            
        return img

    def _draw_landmarks_on_image(self, rgb_image, detection_result):
        """Draws the skeleton and overlays text labels for the body joints."""
        pose_landmarks_list = detection_result.pose_landmarks
        annotated_image = np.copy(rgb_image)
        
        # Get image dimensions to calculate pixel coordinates
        h, w, _ = annotated_image.shape

        pose_landmark_style = drawing_styles.get_default_pose_landmarks_style()
        pose_connection_style = drawing_utils.DrawingSpec(color=(0, 255, 0), thickness=2)

        for pose_landmarks in pose_landmarks_list:
            # 1. Draw the default dots and lines
            drawing_utils.draw_landmarks(
                image=annotated_image,
                landmark_list=pose_landmarks,
                connections=vision.PoseLandmarksConnections.POSE_LANDMARKS,
                landmark_drawing_spec=pose_landmark_style,
                connection_drawing_spec=pose_connection_style)
                
            # 2. Iterate through the coordinates and add text labels
            for idx, landmark in enumerate(pose_landmarks):
                # If the point is in our dictionary (11-32)
                if idx in self.keypoint_names:
                    # Convert normalized 0.0-1.0 coordinate to actual X, Y pixels
                    x, y = int(landmark.x * w), int(landmark.y * h)
                    
                    # Put the text slightly offset (+5px) so it doesn't cover the green dot
                    cv2.putText(
                        annotated_image, 
                        self.keypoint_names[idx], 
                        (x + 5, y - 5), 
                        cv2.FONT_HERSHEY_SIMPLEX, 
                        0.4,           # Font scale
                        (255, 0, 0),   # Color (Red in RGB space)
                        1,             # Line thickness
                        cv2.LINE_AA
                    )

        return annotated_image

In [7]:
def inference_on_video(input_path, output_path):
    cap = cv2.VideoCapture(input_path)

    if not cap.isOpened():
        print(f"❌ Error: Could not open video at {input_path}")
    else:
        detector = PoseDetector(model_asset_path='pose_landmarker_heavy.task')

        frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = int(cap.get(cv2.CAP_PROP_FPS))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))

        progress_bar = tqdm(total=total_frames, desc="Analyzing Squat Form", unit="frames")

        while cap.isOpened():
            success, img = cap.read()
            if not success:
                break 
                
            # Get the actual video timestamp in milliseconds
            timestamp_ms = cap.get(cv2.CAP_PROP_POS_MSEC)
            
            # The API requires strictly increasing timestamps. 
            # If the video restarts or loops, this prevents a crash.
            if timestamp_ms < 0: 
                timestamp_ms = 0

            # Process the frame
            img = detector.find_pose(img, timestamp_ms, draw=True)
            
            out.write(img)
            progress_bar.update(1)

        progress_bar.close()
        cap.release()
        out.release()
        print(f"✅ Done! Processed video saved to: {output_path}")

In [8]:
# UPDATE THIS PATH to your test video
input_path = '/home/rahul/Projects/squat-form-analyzer/sample_vids/youTube_video.mp4' 
output_path = '/home/rahul/Projects/squat-form-analyzer/inferred_vid/youTube_video.mp4'

inference_on_video(input_path, output_path)

I0000 00:00:1775733482.742525  208697 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1775733482.746746  208708 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.2.8-0ubuntu0.25.10.1), renderer: Mesa Intel(R) UHD Graphics 620 (KBL GT2)
W0000 00:00:1775733482.833051  208703 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775733482.959245  208700 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
Analyzing Squat Form: 100%|██████████| 514/514 [01:21<00:00,  6.31frames/s]

✅ Done! Processed video saved to: /home/rahul/Projects/squat-form-analyzer/inferred_vid/youTube_video.mp4


In [9]:
# UPDATE THIS PATH to your test video
input_path = '/home/rahul/Projects/squat-form-analyzer/sample_vids/squat_demo.mp4' 
output_path = '/home/rahul/Projects/squat-form-analyzer/inferred_vid/squat_demo.mp4'

inference_on_video(input_path, output_path)

I0000 00:00:1775733564.482182  210880 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1775733564.484938  210896 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.2.8-0ubuntu0.25.10.1), renderer: Mesa Intel(R) UHD Graphics 620 (KBL GT2)
W0000 00:00:1775733564.562228  210886 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775733564.654197  210888 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
Analyzing Squat Form: 100%|██████████| 514/514 [01:46<00:00,  4.84frames/s]

✅ Done! Processed video saved to: /home/rahul/Projects/squat-form-analyzer/inferred_vid/squat_demo.mp4


# Squat Analyzer

In [10]:
import math
import cv2

class SquatAnalyzer:
    def __init__(self):
        # State variables to track the phase of the squat
        self.is_squatting = False
        
    def calculate_angle(self, p1, p2, p3):
        """
        Calculates the angle between three points.
        p2 is the vertex.
        """
        # Calculate angle in radians
        radians = math.atan2(p3.y - p2.y, p3.x - p2.x) - math.atan2(p1.y - p2.y, p1.x - p2.x)
        angle = abs(radians * 180.0 / math.pi)
        
        # Ensure angle is between 0 and 180 degrees
        if angle > 180.0:
            angle = 360.0 - angle
            
        return angle

    def analyze_pose(self, pose_landmarks):
        """Evaluates the posture and returns feedback."""
        if not pose_landmarks:
            return "No person detected.", False

        # Extract specific right-side landmarks
        # (Using Right Side: Shoulder=12, Hip=24, Knee=26, Ankle=28, Toe=32)
        shoulder = pose_landmarks[12]
        hip = pose_landmarks[24]
        knee = pose_landmarks[26]
        toe = pose_landmarks[32]

        feedback = []
        is_correct = True

        # RULE 1: Hips must descend below the knees
        # Since Y increases downwards, hip.y > knee.y means hips are lower than knees.
        if hip.y < knee.y:
            feedback.append("Lower your hips further.")
            is_correct = False

        # RULE 2: Knees should not significantly extend past toes
        # Assuming person is facing right (X increases to the right)
        # We allow a small margin (e.g., 0.02 normalized units)
        if knee.x > (toe.x + 0.02):
            feedback.append("Maintain knees behind toes.")
            is_correct = False

        # RULE 3: Maintain a relatively vertical back angle
        # Calculate angle between Shoulder, Hip, and a vertical reference point
        vertical_ref = type('Point', (), {'x': hip.x, 'y': 0})() # Point directly above the hip
        back_angle = self.calculate_angle(shoulder, hip, vertical_ref)
        
        # A perfect straight back is 0 degrees. Leaning forward increases the angle.
        if back_angle > 45.0:
            feedback.append("Straighten your back.")
            is_correct = False

        # Compile final feedback string
        if is_correct:
            final_feedback = "✅ Correct Posture"
        else:
            final_feedback = "❌ " + " | ".join(feedback)

        return final_feedback, is_correct

    def draw_feedback(self, img, feedback_text, is_correct):
        """Overlays the feedback text onto the video frame."""
        # Setup colors: Green for correct, Red for incorrect
        color = (0, 255, 0) if is_correct else (0, 0, 255)
        
        # Draw a semi-transparent background rectangle for text readability
        overlay = img.copy()
        cv2.rectangle(overlay, (10, 10), (1200, 60), (0, 0, 0), cv2.FILLED)
        cv2.addWeighted(overlay, 0.6, img, 0.4, 0, img)
        
        # Put the text on the image
        cv2.putText(img, feedback_text, (20, 45), cv2.FONT_HERSHEY_SIMPLEX, 
                    1, color, 2, cv2.LINE_AA)
        
        return img

In [18]:
import cv2
import sys
from tqdm import tqdm

def squat_analyzer_on_video(input_path, output_path):

    cap = cv2.VideoCapture(input_path)

    if not cap.isOpened():
        print(f"❌ Error: Could not open video at {input_path}")
    else:
        # Initialize both of our modules
        detector = PoseDetector(model_asset_path='pose_landmarker_heavy.task')
        analyzer = SquatAnalyzer()

        frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = int(cap.get(cv2.CAP_PROP_FPS))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))

        # --- THE FIX ---
        # Force tqdm to write to stdout and hold its position
        progress_bar = tqdm(
            total=total_frames, 
            desc="Evaluating Squat Form", 
            unit="frames",
            file=sys.stdout, 
            position=0, 
            leave=True
        )

        while cap.isOpened():
            success, img = cap.read()
            if not success:
                break 
                
            timestamp_ms = cap.get(cv2.CAP_PROP_POS_MSEC)
            if timestamp_ms < 0: timestamp_ms = 0

            # 1. Get the Pose Landmarks and draw the skeleton
            img = detector.find_pose(img, timestamp_ms, draw=True)
            
            # 2. Evaluate the posture
            if hasattr(detector, 'results') and detector.results.pose_landmarks:
                landmarks = detector.results.pose_landmarks[0]
                feedback_text, is_correct = analyzer.analyze_pose(landmarks)
                img = analyzer.draw_feedback(img, feedback_text, is_correct)
            else:
                img = analyzer.draw_feedback(img, "Waiting for subject...", False)

            # 3. Write to output
            out.write(img)
            
            # Update the progress bar
            progress_bar.update(1)

        progress_bar.close()
        cap.release()
        out.release()
        print(f"✅ Done! Analyzed video saved to: {output_path}")

# Run the function
# squat_analyzer_on_video('tests/test_videos/sample_squat.mp4', 'demo/squat_analyzed_output.mp4')

In [19]:

input_path = '/home/rahul/Projects/squat-form-analyzer/sample_vids/squat_demo.mp4' 
output_path = '/home/rahul/Projects/squat-form-analyzer/inferred_vid/squat_demo_analyzed.mp4'
squat_analyzer_on_video(input_path, output_path)

I0000 00:00:1775735349.424506  236074 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1775735349.426899  236085 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.2.8-0ubuntu0.25.10.1), renderer: Mesa Intel(R) UHD Graphics 620 (KBL GT2)
W0000 00:00:1775735349.512307  236076 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775735349.596420  236082 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Evaluating Squat Form: 100%|██████████| 514/514 [01:52<00:00,  4.56frames/s]
✅ Done! Analyzed video saved to: /home/rahul/Projects/squat-form-analyzer/inferred_vid/squat_demo_analyzed.mp4


In [20]:

input_path = '/home/rahul/Projects/squat-form-analyzer/sample_vids/youTube_video.mp4' 
output_path = '/home/rahul/Projects/squat-form-analyzer/inferred_vid/youTube_video_analyzed.mp4'
squat_analyzer_on_video(input_path, output_path)

I0000 00:00:1775735462.515550  237783 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1775735462.568412  237794 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.2.8-0ubuntu0.25.10.1), renderer: Mesa Intel(R) UHD Graphics 620 (KBL GT2)
W0000 00:00:1775735462.767921  237789 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775735462.968094  237791 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Evaluating Squat Form: 100%|██████████| 514/514 [02:00<00:00,  4.25frames/s]
✅ Done! Analyzed video saved to: /home/rahul/Projects/squat-form-analyzer/inferred_vid/youTube_video_analyzed.mp4
